# Présidentielles françaises 2002-2022 et tendance 2027

## Équipe et dépôt GitHub
- Manel HASSAN
- Tasnim BOURKHIS (cheffe d'équipe)
- Julia PLAYE-LORENTE
- Marie BARDET
- Inès BENBARA
- Karell Steve JODOM CHOUDJA

Dépôt GitHub : https://github.com/TazBrks/presidentielles-france-2027

Cours : Programming for Data Science (Hugo Alatrista-Salas). Le notebook suit le processus KDD
(sélection, prétraitement, transformation, fouille, interprétation). Toutes les fonctions sont définies d'abord,
puis appelées depuis `main()` à la fin, qui lance aussi le dashboard.

## Le dataset

**Dataset principal : « Données des élections agrégées »** (data.gouv.fr). Il rassemble les résultats officiels
de toutes les élections françaises depuis 1999, au niveau du **bureau de vote**. Deux fichiers :
- `general_results.parquet` : pour chaque bureau et chaque élection, le nombre d'inscrits, de votants,
  d'abstentions, de votes blancs, nuls et exprimés.
- `candidats_results.parquet` : pour chaque bureau, chaque élection et chaque candidat, le nombre de voix.

Nous gardons les **présidentielles de 2002, 2007, 2012, 2017 et 2022** (deux tours chacune).

Les trois dimensions demandées :
- **Temporelle** : cinq élections de 2002 à 2022, puis les sondages 2027 (août à novembre 2026).
- **Spatiale** : bureau de vote, commune, département et région (codes INSEE), avec les contours des départements.
- **Analytique** : voix, abstention, blancs et nuls, regroupés en familles politiques.

**Datasets externes** (pour enrichir et interpréter les résultats) :
- Sondages présidentielle 2027 (data.gouv.fr) : 29 sondages et une projection départementale publiée par un site tiers.
- Sondages 2017, 2022 et 2027, 1er et 2nd tours (tableaux Wikipédia extraits par le dépôt GitHub `hugoC1/sondages_presidentielles`) : pour tester la méthode sur le passé et pour le second tour 2027.
- Liste des communes de France 2026 (data.gouv.fr) : population, superficie, type d'espace, coordonnées.
- INSEE Filosofi 2023 : niveau de vie, pauvreté et inégalités par département.
- INSEE, indice des prix à la consommation (série 001759970) : inflation nationale.
- ACLED (version agrégée) : manifestations et émeutes par région et par semaine depuis 2020.
- Contours des départements : projet open source france-geojson.

In [ ]:
# Bibliothèques
from pathlib import Path
import json
import os
import unicodedata
import zipfile

import numpy as np
import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Dossiers du projet (le notebook peut être lancé depuis la racine ou depuis notebooks/)
DOSSIER_PROJET = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOSSIER_RAW = DOSSIER_PROJET / "data" / "raw"              # données brutes, non versionnées
DOSSIER_REFERENCE = DOSSIER_PROJET / "data" / "reference"  # tables faites à la main (familles, contours)
DOSSIER_PROCESSED = DOSSIER_PROJET / "data" / "processed"  # sorties du notebook, lues par le dashboard

ANNEES = [2002, 2007, 2012, 2017, 2022]                     # présidentielles étudiées
ELECTIONS = [f"{annee}_pres_t{tour}" for annee in ANNEES for tour in (1, 2)]

## Les indicateurs

| Indicateur | Calcul | Lecture |
|---|---|---|
| **Part des exprimés** d'une famille | voix des candidats de la famille / suffrages exprimés | 0,30 = 30 % des votes valides |
| **Taux d'abstention** | abstentions / inscrits | part des inscrits qui n'ont pas voté |
| **Taux de blancs et nuls** | (blancs + nuls) / votants | part des votants qui n'ont choisi aucun candidat |
| **Écart au national** | part dans le département − part en France | positif : la famille fait mieux que sa moyenne nationale |
| **Tendance 2027** | moyenne pondérée des sondages + écart au national de 2022 (amortissement choisi par le test sur le passé) | où chaque famille ferait plus ou moins que sa moyenne nationale ; ce n'est pas une prévision |
| **Erreur moyenne** (test sur le passé) | moyenne des écarts absolus entre part projetée et part réelle, en points | 1,2 = on se trompe de 1,2 point en moyenne par département et par famille |
| **Demi-vie** d'un sondage | poids = 0,5 ^ (âge en jours / 30) | un sondage d'il y a 30 jours compte deux fois moins que le plus récent |
| **Corrélation** (Pearson) | entre -1 et 1 | proche de 1 ou de -1 : lien fort ; proche de 0 : pas de lien linéaire |
| **R²** d'une régression | part de la variance expliquée | 0,5 = le modèle explique la moitié des différences entre départements |
| **Support** d'une règle | part des départements qui ont tous les items de la règle | 0,25 = un département sur quatre |
| **Confiance** d'une règle A → B | support(A et B) / support(A) | 0,8 = 80 % des départements qui ont A ont aussi B |
| **Lift** d'une règle | confiance / support(B) | au-dessus de 1 : A rend B plus fréquent que la moyenne |
| **Silhouette** d'un clustering | entre -1 et 1 | plus elle est haute, plus les groupes sont nets |
| **Cohérence spatiale** d'un clustering | part des 4 départements les plus proches (distance entre centres) qui sont dans le même groupe | 0,8 = 80 % des voisins sont dans le même groupe : les groupes forment des zones d'un seul tenant |
| **Niveau de vie médian** (INSEE) | revenu disponible par unité de consommation qui partage la population en deux moitiés, en € par an | plus il est haut, plus le département est aisé |
| **Taux de pauvreté** (INSEE) | part des habitants sous 60 % du niveau de vie médian national | 0,15 = 15 % des habitants sont pauvres |
| **Indice de Gini** (INSEE) | inégalité des niveaux de vie, entre 0 et 1 | 0 = tout le monde a le même revenu ; plus il est haut, plus c'est inégal |
| **Part du chômage dans le revenu** (INSEE) | allocations chômage / revenu disponible total | plus elle est haute, plus le chômage pèse dans le département |
| **Densité** | population / superficie, en habitants par km² | distingue les départements urbains et ruraux |
| **Part de population rurale** | habitants des communes « rurales » ou « bourgs » (grille de densité INSEE) / population | 0,6 = 60 % des habitants vivent en zone rurale |
| **Inflation sur 12 mois** (INSEE) | indice des prix du mois / indice du même mois un an avant − 1 | 0,05 = les prix ont monté de 5 % en un an |
| **Manifestations pour 100 000 habitants** (ACLED) | manifestations et émeutes dans la région pendant les 12 mois avant l'élection / population de la région × 100 000 | mesure le climat social ; disponible seulement depuis 2020 |
| **Erreur nationale** (test complet) | part annoncée par les sondages de l'époque − part réelle, en points | +8 = la famille était surestimée de 8 points ; négatif = sous-estimée |
| **Famille en tête correcte** (test complet) | part des départements où la famille prévue en tête est bien arrivée en tête | 0,85 = le bon vainqueur dans 85 % des départements |
| **Score du RN par duel** (second tour 2027) | moyenne pondérée (demi-vie de 30 jours) du score de Le Pen dans les sondages d'un même duel | 55 = Le Pen à 55 % des exprimés face à cet adversaire |
| **Part du RN par département** (second tour 2027) | score du RN par duel + écart du département au national au second tour 2022 | au-dessus de 0,5 : Le Pen en tête dans le département |
| **Biais des sondages de second tour** | score de Le Pen dans les sondages − score réel, en points (2017 et 2022) | +3 = les sondages la surestimaient de 3 points |

**Familles politiques** : chaque candidat est rangé dans une des 9 familles (extrême gauche, gauche radicale,
gauche, écologistes, centre, droite, souverainistes, extrême droite, divers), avec les tables
`data/reference/candidats_familles.csv` (2002-2022) et `candidats_familles_2027.csv`. Cela permet de comparer
des élections où les candidats changent.

## Étape 1 : sélection des données (chargement)

In [ ]:
# Fonction qui charge les résultats généraux des présidentielles
# Entrée : chemin du fichier Parquet, liste des élections à garder (ex. "2022_pres_t1")
# Sortie : DataFrame, une ligne par bureau de vote et par élection (inscrits, votants, abstentions...)
def charger_resultats_generaux(chemin=DOSSIER_RAW / "general_results.parquet", elections=ELECTIONS):
    table = pq.read_table(chemin, filters=[("id_election", "in", elections)])  # filtre à la lecture : on ne charge que les présidentielles
    return table.to_pandas()


# Fonction qui charge les voix par candidat des présidentielles
# Entrée : chemin du fichier Parquet, liste des élections à garder
# Sortie : DataFrame, une ligne par bureau, élection et candidat
def charger_resultats_candidats(chemin=DOSSIER_RAW / "candidats_results.parquet", elections=ELECTIONS):
    # On ne garde que les colonnes utiles : les colonnes « liste », « binome »... servent aux autres élections
    colonnes = ["id_election", "code_departement", "code_commune", "code_bv",
                "nom", "prenom", "voix", "ratio_voix_inscrits", "ratio_voix_exprimes"]
    table = pq.read_table(chemin, columns=colonnes, filters=[("id_election", "in", elections)])
    return table.to_pandas()


# Fonction qui charge la table candidat -> famille politique (2002-2022)
# Entrée : chemin du CSV
# Sortie : DataFrame (annee, nom, prenom, famille)
def charger_familles(chemin=DOSSIER_REFERENCE / "candidats_familles.csv"):
    return pd.read_csv(chemin)


# Fonction qui lit un CSV des fichiers sondages 2027
# Entrée : nom du fichier dans data/raw
# Sortie : DataFrame (séparateur ; et virgule décimale gérés)
def lire_csv_sondage(nom_fichier):
    return pd.read_csv(DOSSIER_RAW / nom_fichier, sep=";", decimal=",", encoding="utf-8-sig",
                       dtype={"code_departement": str})


# Fonction qui charge les 29 sondages 2027
# Entrée : chemin du fichier sondages.json
# Sortie : DataFrame, une ligne par sondage, hypothèse et candidat (intentions_pct)
def charger_sondages(chemin=DOSSIER_RAW / "sondages.json"):
    with open(chemin, encoding="utf-8") as f:
        brut = json.load(f)
    lignes = []
    for i, sondage in enumerate(brut["sondages"]):                 # un sondage contient plusieurs hypothèses
        for j, hypothese in enumerate(sondage["hypotheses"]):      # une hypothèse = une liste de candidats testés
            for slug, score in hypothese["scores"].items():
                lignes.append({"id_sondage": i, "institut": sondage["institut"],
                               "terrain_fin": sondage["terrain_fin"], "id_hypothese": j,
                               "slug": slug, "intentions_pct": score})
    return pd.DataFrame(lignes)


# Fonction qui charge la liste des communes 2026
# Entrée : chemin du fichier Parquet
# Sortie : DataFrame (code INSEE, département, région, population, superficie, type d'espace, coordonnées)
def charger_communes(chemin=DOSSIER_RAW / "communes-france-2026.parquet"):
    colonnes = ["code_insee", "dep_code", "reg_nom", "population", "superficie_km2",
                "grille_densite_texte", "latitude_centre", "longitude_centre"]
    return pd.read_parquet(chemin, columns=colonnes)


INDICATEURS_FILOSOFI = {"MED_SL": "niveau_vie_median", "PR_MD60": "taux_pauvrete",
                        "GI_SL": "indice_gini", "S_EI_DI_UNE": "part_revenu_chomage",
                        "S_SOC_BEN_DI_MIN_SOC": "part_revenu_minima_sociaux",
                        "S_RET_PEN_DI": "part_revenu_retraites"}


# Fonction qui charge les indicateurs Filosofi 2023 (INSEE) par département
# Entrée : chemin du zip téléchargé sur insee.fr
# Sortie : DataFrame, une ligne par département, une colonne par indicateur de INDICATEURS_FILOSOFI
def charger_filosofi(chemin=DOSSIER_RAW / "FILOSOFI_CC_csv.zip"):
    with zipfile.ZipFile(chemin) as z:
        nom = next(n for n in z.namelist() if n.endswith("_data.csv"))
        with z.open(nom) as f:
            brut = pd.read_csv(f, sep=";", dtype={"GEO": str})
    # Le niveau commune est presque vide (secret statistique) : on garde le niveau département
    dep = brut[(brut["GEO_OBJECT"] == "DEP") & brut["FILOSOFI_MEASURE"].isin(INDICATEURS_FILOSOFI)]
    table = dep.pivot_table(index="GEO", columns="FILOSOFI_MEASURE", values="OBS_VALUE")
    return table.rename(columns=INDICATEURS_FILOSOFI).reset_index().rename(columns={"GEO": "code_departement"})


# Fonction qui charge l'export ACLED (version agrégée)
# Entrée : chemin du CSV
# Sortie : DataFrame (semaine, région, type d'événement, nombre d'événements), France seulement
def charger_acled(chemin=DOSSIER_RAW / "acled_france_2020_2026.csv"):
    acled = pd.read_csv(chemin, parse_dates=["week"])
    acled = acled[acled["country"] == "France"].copy()      # on retire les quelques lignes « Atlantic Ocean »
    acled["cle_region"] = acled["admin1"].map(normaliser_nom)
    return acled


# Fonction qui charge l'indice des prix (INSEE) et calcule l'inflation sur 12 mois
# Entrée : chemin du zip de la série 001759970
# Sortie : DataFrame indexé par mois (ipc, inflation_12_mois)
def charger_ipc(chemin=DOSSIER_RAW / "serie_001759970_23112026.zip"):
    with zipfile.ZipFile(chemin) as z:
        with z.open("valeurs_mensuelles.csv") as f:
            ipc = pd.read_csv(f, sep=";", skiprows=4, header=None, usecols=[0, 1], names=["mois", "ipc"])
    ipc["mois"] = pd.to_datetime(ipc["mois"])
    ipc = ipc.sort_values("mois").set_index("mois")
    ipc["inflation_12_mois"] = ipc["ipc"].pct_change(12)   # variation de l'indice sur un an
    return ipc


# Fonction qui charge les contours des départements de métropole
# Entrée : chemin du GeoJSON
# Sortie : dictionnaire GeoJSON (propriété « code » = code du département)
def charger_contours(chemin=DOSSIER_REFERENCE / "departements.geojson"):
    with open(chemin, encoding="utf-8") as f:
        return json.load(f)

## Étape 2 : exploration des données

Pour chaque table : nombre de lignes et de colonnes, type de chaque colonne, valeurs manquantes,
et plage de valeurs (minimum et maximum pour les nombres, nombre de valeurs distinctes pour le texte).

In [ ]:
# Fonction qui résume un DataFrame
# Entrée : DataFrame, nom à afficher
# Sortie : DataFrame avec une ligne par colonne (type, valeurs manquantes, minimum, maximum, valeurs distinctes)
def explorer(df, nom):
    print(f"{nom} : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
    resume = pd.DataFrame({
        "type": df.dtypes.astype(str),
        "manquants": df.isna().sum(),
        "manquants_pct": (df.isna().mean() * 100).round(2),
        "valeurs_distinctes": df.nunique(),
    })
    numeriques = df.select_dtypes("number")
    resume["minimum"] = numeriques.min()
    resume["maximum"] = numeriques.max()
    return resume

## Étape 3 : prétraitement

Décisions :
- On garde seulement les **présidentielles** (le fichier contient 56 élections : européennes, législatives...).
- On retire les colonnes qui ne servent qu'aux autres élections (listes, binômes, têtes de liste).
- La colonne `nuance` (étiquette politique) est vide pour 2017 et 2022 : on la remplace par notre table des familles.
- Les noms sont comparés sans accents ni majuscules, car l'écriture change d'une année à l'autre (MÉLENCHON, MELENCHON).
- Les cartes montrent la métropole ; l'outre-mer et les Français de l'étranger restent dans les totaux nationaux.

In [ ]:
# Fonction qui met un nom en majuscules sans accents
# Entrée : un nom (texte)
# Sortie : le nom normalisé, pour comparer les années entre elles
def normaliser_nom(nom):
    sans_accents = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    return sans_accents.upper().strip()


# Fonction qui ajoute l'année et le tour
# Entrée : DataFrame avec une colonne id_election (ex. "2022_pres_t1")
# Sortie : copie du DataFrame avec les colonnes annee et tour
def ajouter_annee_tour(df):
    df = df.copy()
    df["annee"] = df["id_election"].str[:4].astype(int)
    df["tour"] = df["id_election"].str[-1].astype(int)
    return df


# Fonction qui associe chaque candidat à sa famille politique
# Entrée : résultats par candidat (avec annee), table des familles
# Sortie : résultats avec une colonne famille (affiche les candidats sans famille s'il y en a)
def ajouter_familles(candidats, familles):
    familles = familles.assign(cle_nom=familles["nom"].map(normaliser_nom))[["annee", "cle_nom", "famille"]]
    candidats = candidats.assign(cle_nom=candidats["nom"].map(normaliser_nom))
    resultat = candidats.merge(familles, on=["annee", "cle_nom"], how="left").drop(columns="cle_nom")
    manquants = resultat.loc[resultat["famille"].isna(), "nom"].unique()
    if len(manquants) > 0:
        print("Candidats sans famille politique :", list(manquants))
    return resultat

## Étape 4 : transformation (agrégation par département)

In [ ]:
# Fonction qui agrège les résultats généraux par département
# Entrée : résultats généraux par bureau de vote (avec annee et tour)
# Sortie : une ligne par élection et département, avec taux d'abstention et de blancs et nuls
def agreger_par_departement(generaux):
    colonnes = ["inscrits", "abstentions", "votants", "blancs", "nuls", "exprimes"]
    dep = (generaux.groupby(["id_election", "annee", "tour", "code_departement", "libelle_departement"])[colonnes]
           .sum().reset_index())
    dep["taux_abstention"] = dep["abstentions"] / dep["inscrits"]
    dep["taux_blancs_nuls"] = (dep["blancs"] + dep["nuls"]) / dep["votants"]
    return dep


# Fonction qui calcule la part de chaque famille par département
# Entrée : résultats par candidat (avec famille), résultats généraux par département
# Sortie : une ligne par élection, département et famille (voix, exprimés, part_exprimes)
def voix_par_famille_departement(candidats, generaux_dep):
    voix = (candidats.groupby(["id_election", "annee", "tour", "code_departement", "famille"])["voix"]
            .sum().reset_index())
    voix = voix.merge(generaux_dep[["id_election", "code_departement", "exprimes"]],
                      on=["id_election", "code_departement"], how="left")
    voix["part_exprimes"] = voix["voix"] / voix["exprimes"]
    return voix


# Fonction qui construit la grille complète département x famille x année au 1er tour
# Entrée : parts par famille et département
# Sortie : DataFrame avec 0 quand une famille n'avait pas de candidat cette année-là
def parts_premier_tour(familles_dep):
    t1 = familles_dep[familles_dep["tour"] == 1]
    grille = pd.MultiIndex.from_product([t1["code_departement"].unique(), t1["famille"].unique(), ANNEES],
                                        names=["code_departement", "famille", "annee"])
    parts = t1.set_index(["code_departement", "famille", "annee"])["part_exprimes"].reindex(grille, fill_value=0)
    return parts.reset_index()


# Fonction qui calcule la part nationale de chaque famille au 1er tour
# Entrée : résultats par candidat (avec famille), résultats généraux
# Sortie : une ligne par année et famille (part_nationale)
def parts_nationales(candidats, generaux):
    t1 = candidats[candidats["tour"] == 1]
    voix = t1.groupby(["annee", "famille"])["voix"].sum()
    exprimes = generaux[generaux["tour"] == 1].groupby("annee")["exprimes"].sum()
    return (voix / exprimes.reindex(voix.index.get_level_values("annee")).values).rename("part_nationale").reset_index()


# Fonction qui résume les communes par département
# Entrée : liste des communes
# Sortie : une ligne par département (région, population, densité, part de population en commune rurale)
def profil_departements(communes):
    communes = communes.assign(rural=communes["grille_densite_texte"].str.startswith(("Rural", "Bourgs")))
    communes["population_rurale"] = communes["population"].where(communes["rural"], 0)
    profil = communes.groupby("dep_code").agg(
        reg_nom=("reg_nom", "first"), population=("population", "sum"),
        superficie_km2=("superficie_km2", "sum"), population_rurale=("population_rurale", "sum"),
        latitude=("latitude_centre", "mean"), longitude=("longitude_centre", "mean"))
    profil["densite"] = profil["population"] / profil["superficie_km2"]
    profil["part_population_rurale"] = profil["population_rurale"] / profil["population"]
    return profil.drop(columns="population_rurale").reset_index().rename(columns={"dep_code": "code_departement"})


# Fonction qui compte les événements ACLED par région sur une période
# Entrée : table ACLED, date de début, date de fin, types d'événements gardés
# Sortie : une ligne par région (nombre de manifestations et d'émeutes)
def acled_par_region(acled, debut, fin, types=("Protests", "Riots")):
    periode = acled[(acled["week"] >= debut) & (acled["week"] < fin) & acled["event_type"].isin(types)]
    table = periode.pivot_table(index="cle_region", columns="event_type", values="events",
                                aggfunc="sum", fill_value=0)
    return table.rename(columns=lambda t: "evenements_" + t.lower()).reset_index()


# Fonction qui assemble une ligne par département pour une élection
# Entrée : parts par famille, résultats généraux par département, profil des communes, Filosofi, ACLED par région
# Sortie : DataFrame (vote par famille, abstention, indicateurs socio-économiques, ACLED pour 100 000 habitants)
def table_departements(familles_dep, generaux_dep, profil, filosofi, acled_region, election="2022_pres_t1"):
    vote = (familles_dep[familles_dep["id_election"] == election]
            .pivot_table(index="code_departement", columns="famille", values="part_exprimes", fill_value=0)
            .add_prefix("vote_"))
    abstention = generaux_dep[generaux_dep["id_election"] == election].set_index("code_departement")["taux_abstention"]
    table = vote.join(abstention).reset_index()
    # jointure interne : les codes des départements d'outre-mer diffèrent entre les deux sources (ZA / 971),
    # le tableau porte donc sur les 96 départements de métropole
    table = table.merge(profil, on="code_departement", how="inner").merge(filosofi, on="code_departement", how="left")
    table["cle_region"] = table["reg_nom"].map(normaliser_nom)
    table = table.merge(acled_region, on="cle_region", how="left").drop(columns="cle_region")
    population_region = table.groupby("reg_nom")["population"].transform("sum")
    for col in [c for c in table.columns if c.startswith("evenements_")]:
        table[col + "_pour_100k"] = table[col] / population_region * 1e5   # ACLED est régional : rapporté à la population de la région
    return table

### Une seule table pour toutes les sources : les clés communes

Toutes les sources se relient entre elles par des **clés communes**. On les rassemble dans une seule table,
avec **une ligne par département et par tour d'élection** (2002 à 2022, plus la tendance 2027).

| Source | Niveau d'origine | Clé utilisée | Comment on la relie |
|---|---|---|---|
| Résultats (data.gouv.fr) | bureau de vote | `code_departement`, `code_commune` (INSEE), `id_election` | agrégés au département, `id_election` donne l'année et le tour |
| Familles politiques (table maison) | candidat | `annee` + nom normalisé | rattache chaque candidat à une famille |
| Communes 2026 (data.gouv.fr) | commune | `code_insee` → `dep_code`, `reg_nom` | agrégées au département : population, densité, coordonnées, région |
| Filosofi 2023 (INSEE) | département | `GEO` = code du département | jointure directe sur `code_departement` |
| ACLED | région × semaine | nom de la région (normalisé) + date | relié par la région du département (via les communes) et les 12 mois avant l'élection |
| Indice des prix (INSEE) | France × mois | mois | relié par la date de l'élection (inflation sur 12 mois) |
| Sondages 2027 | France × famille | `famille` + année | donnent le niveau national de la tendance 2027 |
| Contours (france-geojson) | département | `code` = code du département | cartes |

La fonction `verifier_cles` compte, pour chaque jointure, combien de départements trouvent une correspondance :
c'est la preuve que les clés sont bien communes. Seuls les départements de métropole ont toutes les sources
(les codes d'outre-mer diffèrent : `ZA` dans les résultats, `971` à l'INSEE).

In [ ]:
DATES_ELECTIONS = {  # date de chaque tour, pour relier l'inflation et ACLED à l'élection
    "2002_pres_t1": "2002-04-21", "2002_pres_t2": "2002-05-05", "2007_pres_t1": "2007-04-22",
    "2007_pres_t2": "2007-05-06", "2012_pres_t1": "2012-04-22", "2012_pres_t2": "2012-05-06",
    "2017_pres_t1": "2017-04-23", "2017_pres_t2": "2017-05-07", "2022_pres_t1": "2022-04-10",
    "2022_pres_t2": "2022-04-24",
}


# Fonction qui compte, pour chaque jointure, les départements qui trouvent une correspondance
# Entrée : résultats par département, profil des communes, Filosofi, ACLED, contours
# Sortie : DataFrame (jointure, clé, départements côté résultats, départements trouvés, part trouvée)
def verifier_cles(generaux_dep, profil, filosofi, acled, contours):
    deps = set(generaux_dep["code_departement"])
    regions_acled = set(acled["cle_region"])
    regions_dep = profil.set_index("code_departement")["reg_nom"].map(normaliser_nom)
    controles = {
        ("Résultats → communes", "code_departement = dep_code"): deps & set(profil["code_departement"]),
        ("Résultats → Filosofi", "code_departement = GEO"): deps & set(filosofi["code_departement"]),
        ("Résultats → contours", "code_departement = code"): deps & {f["properties"]["code"] for f in contours["features"]},
        ("Résultats → ACLED", "région du département = admin1"): {d for d in deps if regions_dep.get(d) in regions_acled},
    }
    lignes = [{"jointure": j, "cle": c, "departements_resultats": len(deps), "departements_trouves": len(t)}
              for (j, c), t in controles.items()]
    table = pd.DataFrame(lignes)
    table["part_trouvee"] = table["departements_trouves"] / table["departements_resultats"]
    return table


# Fonction qui rassemble toutes les sources dans une seule table
# Entrée : voix par famille et département, résultats par département, profil des communes, Filosofi, ACLED,
#          indice des prix, tendance 2027
# Sortie : DataFrame, une ligne par département et par tour (2002-2022) plus la tendance 2027 au 1er tour,
#          avec le vote, la participation, le profil, le revenu, l'inflation et les manifestations ; métropole
def construire_table_unique(familles_dep, generaux_dep, profil, filosofi, acled, ipc, tendance):
    # 1. Résultats : une ligne par département et tour, une colonne par famille (clé : code_departement + id_election)
    vote = (familles_dep.pivot_table(index=["code_departement", "id_election"], columns="famille",
                                     values="part_exprimes", fill_value=0).add_prefix("vote_").reset_index())
    participation = generaux_dep[["code_departement", "id_election", "inscrits", "taux_abstention", "taux_blancs_nuls"]]
    table = vote.merge(participation, on=["code_departement", "id_election"], how="left")
    table = ajouter_annee_tour(table)
    table["date_election"] = pd.to_datetime(table["id_election"].map(DATES_ELECTIONS))
    # 2. Tendance 2027 (1er tour), avec les mêmes colonnes de vote
    t27 = (tendance.pivot_table(index="code_departement", columns="famille", values="part_2027", fill_value=0)
           .add_prefix("vote_").reset_index().assign(id_election="2027_tendance_t1", annee=2027, tour=1))
    table = pd.concat([table, t27], ignore_index=True)
    table["est_tendance"] = table["annee"] == 2027
    # 3. Communes et Filosofi (clé : code_departement) ; jointure interne = métropole seulement
    table = table.merge(profil, on="code_departement", how="inner")
    table = table.merge(filosofi, on="code_departement", how="left")
    # 4. Inflation sur 12 mois le mois de l'élection (clé : date -> mois)
    mois = table["date_election"].dt.to_period("M").dt.to_timestamp()
    table["inflation_12_mois"] = mois.map(ipc["inflation_12_mois"])
    # 5. ACLED : manifestations et émeutes dans la région pendant les 12 mois avant l'élection (clé : région + date)
    table["cle_region"] = table["reg_nom"].map(normaliser_nom)
    evenements = acled[acled["event_type"].isin(["Protests", "Riots"])]
    debut_acled = acled["week"].min()
    def manifestations(ligne):
        if pd.isna(ligne["date_election"]) or ligne["date_election"] - pd.DateOffset(years=1) < debut_acled:
            return np.nan   # ACLED ne couvre la France que depuis 2020
        periode = evenements[(evenements["cle_region"] == ligne["cle_region"])
                             & (evenements["week"] >= ligne["date_election"] - pd.DateOffset(years=1))
                             & (evenements["week"] < ligne["date_election"])]
        return periode["events"].sum()
    table["manifestations_region_12_mois"] = table.apply(manifestations, axis=1)
    population_region = table.groupby(["id_election", "reg_nom"])["population"].transform("sum")
    table["manifestations_pour_100k"] = table["manifestations_region_12_mois"] / population_region * 1e5
    return table.drop(columns="cle_region")

## Étape 5 : fouille de données

### Requête 1 (regroupement) : qui arrive en tête, et comment les familles évoluent

On somme les voix par candidat et par élection (`groupby` + `sum`) pour retrouver les quatre premiers de chaque
élection, puis on calcule la part nationale de chaque famille de 2002 à 2022 (2027 : moyenne des sondages).
On mesure aussi l'abstention nationale et les départements où une famille a le plus progressé.

In [ ]:
# Fonction qui donne les quatre premiers candidats de chaque élection
# Entrée : résultats par candidat
# Sortie : DataFrame (élection, nom, voix totales), quatre lignes par élection
def resume_national(candidats):
    totaux = candidats.groupby(["id_election", "nom"])["voix"].sum().reset_index()
    return (totaux.sort_values(["id_election", "voix"], ascending=[True, False])
            .groupby("id_election").head(4))


# Fonction qui construit l'évolution nationale des familles
# Entrée : parts nationales 2002-2022, intentions 2027 par famille
# Sortie : DataFrame année x famille (2027 = moyenne des sondages)
def evolution_nationale(parts_nat, intentions_famille):
    table = parts_nat.pivot_table(index="annee", columns="famille", values="part_nationale", fill_value=0)
    table.loc[2027] = intentions_famille.reindex(table.columns).fillna(0)
    return table


# Fonction qui trace l'évolution nationale des familles
# Entrée : table année x famille, liste de familles à afficher (toutes par défaut)
# Sortie : figure Plotly
def graphique_evolution(evolution, familles=None):
    familles = familles or list(evolution.columns)
    df = evolution[familles].reset_index().melt(id_vars="annee", var_name="famille", value_name="part")
    fig = px.line(df, x="annee", y="part", color="famille", markers=True,
                  title="Part des exprimés au 1er tour (2027 : moyenne des sondages)")
    fig.add_vline(x=2024.5, line_dash="dot")   # sépare les résultats réels des sondages
    fig.update_yaxes(tickformat=".0%")
    return fig


# Fonction qui calcule l'abstention nationale
# Entrée : résultats généraux par département
# Sortie : DataFrame année x tour (taux d'abstention)
def evolution_abstention(generaux_dep):
    nat = generaux_dep.groupby(["annee", "tour"])[["abstentions", "inscrits"]].sum()
    return (nat["abstentions"] / nat["inscrits"]).unstack("tour").rename(columns=lambda t: f"tour_{t}")


# Fonction qui trouve les départements où une famille a le plus et le moins progressé
# Entrée : grille des parts au 1er tour, famille, années de début et de fin, nombre de départements
# Sortie : DataFrame (parts des deux années, évolution en points), n plus fortes puis n plus faibles
def departements_qui_bougent(parts_dep, famille, annee_debut=2017, annee_fin=2022, n=5):
    metropole = ~parts_dep["code_departement"].str.startswith("Z")
    df = parts_dep[metropole & (parts_dep["famille"] == famille) & parts_dep["annee"].isin([annee_debut, annee_fin])]
    table = df.pivot_table(index="code_departement", columns="annee", values="part_exprimes")
    table["evolution_points"] = 100 * (table[annee_fin] - table[annee_debut])
    table = table.sort_values("evolution_points")
    return pd.concat([table.tail(n)[::-1], table.head(n)])


# Fonction qui estime ce qui explique le vote d'une famille par département
# Entrée : table des départements, colonne à expliquer, liste des indicateurs
# Sortie : R² du modèle, coefficients (effet d'un écart-type de chaque indicateur, en points)
def modele_explicatif(departements, cible, indicateurs):
    donnees = departements[[cible] + indicateurs].dropna()
    X = StandardScaler().fit_transform(donnees[indicateurs])   # standardisé : les coefficients sont comparables
    modele = LinearRegression().fit(X, donnees[cible])
    coefficients = pd.Series(modele.coef_ * 100, index=indicateurs, name="effet_en_points").sort_values()
    return modele.score(X, donnees[cible]), coefficients

### Requête 2 (motifs fréquents) : règles d'association entre profils de départements

Chaque département de métropole devient une « transaction » dont les items décrivent son profil en 2022 :
par exemple « vote extrême droite élevé » si la part de l'extrême droite y est au-dessus de la médiane des départements,
« pauvreté élevée », « rural », « niveau de vie bas »... On cherche ensuite les **motifs fréquents** avec
l'algorithme **Apriori**, puis les **règles d'association** du type *{pauvreté élevée, rural} → {vote extrême droite élevé}*.

On garde les règles dont la conséquence est un comportement de vote, avec un support d'au moins 20 %,
une confiance d'au moins 70 % et un lift supérieur à 1,3. Le seuil « élevé » est la médiane : chaque item
concerne donc la moitié des départements, ce qui donne un support de base de 0,5.

In [ ]:
ITEMS = {  # colonne de la table des départements -> (nom de l'item, True si « élevé » = au-dessus de la médiane)
    "vote_extreme_droite": ("vote extrême droite élevé", True),
    "vote_gauche_radicale": ("vote gauche radicale élevé", True),
    "vote_centre": ("vote centre élevé", True),
    "taux_abstention": ("abstention élevée", True),
    "taux_pauvrete": ("pauvreté élevée", True),
    "niveau_vie_median": ("niveau de vie bas", False),
    "part_population_rurale": ("rural", True),
    "part_revenu_chomage": ("chômage élevé", True),
    "evenements_protests_pour_100k": ("manifestations élevées", True),
}
ITEMS_VOTE = {"vote extrême droite élevé", "vote gauche radicale élevé", "vote centre élevé", "abstention élevée",
              "extrême droite en forte hausse"}


# Fonction qui transforme chaque département en transaction d'items
# Entrée : table des départements, grille des parts (pour l'évolution 2017-2022 de l'extrême droite)
# Sortie : DataFrame booléen, une ligne par département, une colonne par item
def construire_transactions(departements, parts_dep):
    transactions = pd.DataFrame(index=departements["code_departement"])
    for colonne, (item, au_dessus) in ITEMS.items():
        valeurs = departements.set_index("code_departement")[colonne]
        mediane = valeurs.median()
        transactions[item] = valeurs > mediane if au_dessus else valeurs < mediane
    hausse = departements_qui_bougent(parts_dep, "extreme_droite", n=len(departements))["evolution_points"]
    hausse = hausse[~hausse.index.duplicated()]
    transactions["extrême droite en forte hausse"] = hausse.reindex(transactions.index) > hausse.median()
    return transactions


# Fonction qui extrait les règles d'association avec Apriori
# Entrée : transactions booléennes, support, confiance et lift minimums
# Sortie : DataFrame des règles (antécédents, conséquent, support, confiance, lift), triées par lift
def regles_association(transactions, support_min=0.2, confiance_min=0.7, lift_min=1.3):
    motifs = apriori(transactions, min_support=support_min, use_colnames=True)
    regles = association_rules(motifs, metric="confidence", min_threshold=confiance_min)
    # on garde les règles qui expliquent un comportement de vote à partir du profil du département
    regles = regles[(regles["lift"] >= lift_min)
                    & regles["consequents"].map(lambda c: len(c) == 1 and next(iter(c)) in ITEMS_VOTE)
                    & regles["antecedents"].map(lambda a: not (a & ITEMS_VOTE))]
    regles = regles.assign(antecedents=regles["antecedents"].map(lambda a: " + ".join(sorted(a))),
                           consequents=regles["consequents"].map(lambda c: next(iter(c))))
    return (regles[["antecedents", "consequents", "support", "confidence", "lift"]]
            .sort_values("lift", ascending=False).reset_index(drop=True))

### Requête 3 (temporelle) : tendance 2027 par département

Méthode de prévision :
1. Pour chaque département et chaque famille, on calcule l'**écart au national** à chaque 1er tour de 2002 à 2022.
2. On prolonge cet écart jusqu'en 2027 avec la pente d'une **régression linéaire**, **amortie** : on ne garde
   qu'une fraction `a` de la pente (a = 0 : on garde l'écart de 2022 tel quel ; a = 1 : droite complète).
3. On l'ajoute à la part nationale de la famille dans la **moyenne pondérée des sondages 2027** :
   - les hypothèses qui testent un candidat qui a renoncé (Faure, Royal, Guedj, Maurel...) sont retirées ;
   - chaque hypothèse compte pareil dans son sondage ;
   - un sondage plus vieux de 30 jours que le plus récent pèse deux fois moins (demi-vie de 30 jours).
4. Les scores négatifs sont ramenés à 0, puis chaque département est remis à 100 %.

**Test sur le passé (validation)** : on applique la même méthode pour « prévoir » 2017 avec 2002-2012, et 2022 avec
2002-2017, en lui donnant le vrai résultat national. L'**erreur moyenne** (en points, par département et par famille)
dit à quel point la répartition géographique est fiable ; on choisit l'amortissement qui se trompe le moins.

Résultat : l'erreur moyenne passe de 1,2 point (a = 0) à 1,4 point (a = 1). Prolonger la pente **dégrade** la
prévision : la géographie du vote bouge peu d'une élection à l'autre, et la meilleure méthode est de garder l'écart
de 2022 (ce que les politologues appellent un *swing uniforme*). Avec le vrai résultat national, on se trompe donc
d'environ 1 point par département.
L'incertitude principale reste donc le niveau national, c'est-à-dire les sondages, faits 5 mois avant l'élection.

C'est une **tendance**, pas une prévision. On la compare aussi à la projection départementale publiée avec les sondages
(site Repère 2027), qui repose sur un seul sondage.

In [ ]:
# Fonction qui liste les candidats qui ont renoncé à se présenter en 2027
# Entrée : nom du fichier des candidatures, chemin de la table des familles 2027
# Sortie : ensemble des identifiants (slug) des candidats retirés
def candidats_retires(nom_fichier="candidatures-presidentielle-2027.csv",
                      chemin=DOSSIER_REFERENCE / "candidats_familles_2027.csv"):
    candidatures = lire_csv_sondage(nom_fichier)
    retires = {normaliser_nom(n).split(" ", 1)[-1] for n in candidatures.loc[candidatures["statut_code"] == "withdrawn", "nom"]}
    table = pd.read_csv(chemin)
    return set(table.loc[table["nom"].map(normaliser_nom).isin(retires), "slug"])


# Fonction qui calcule la moyenne pondérée des sondages 2027
# Entrée : sondages (une ligne par sondage, hypothèse et candidat), candidats retirés,
#          demi-vie en jours (un sondage plus vieux de demi_vie jours pèse deux fois moins), chemin de la table des familles
# Sortie : intentions moyennes par candidat (avec famille), intentions par famille (proportions, total = 1)
def moyenne_sondages(sondages, retires=frozenset(), demi_vie=30,
                     chemin=DOSSIER_REFERENCE / "candidats_familles_2027.csv"):
    table = pd.read_csv(chemin)
    manquants = set(sondages["slug"]) - set(table["slug"])
    if manquants:
        print("Candidats 2027 sans famille :", sorted(manquants))
    # 1. On retire les hypothèses qui testent un candidat qui a renoncé : elles ne décrivent plus l'élection possible
    hyp_retirees = sondages.loc[sondages["slug"].isin(retires), ["id_sondage", "id_hypothese"]].drop_duplicates()
    cle = ["id_sondage", "id_hypothese"]
    sondages = sondages.merge(hyp_retirees.assign(retiree=True), on=cle, how="left")
    sondages = sondages[sondages["retiree"].isna()].drop(columns="retiree")
    # 2. Grille complète : un candidat absent d'une hypothèse y compte pour 0 ; moyenne des hypothèses de chaque sondage
    grille = sondages.pivot_table(index=cle, columns="slug", values="intentions_pct", fill_value=0)
    par_sondage = grille.groupby(level="id_sondage").mean()
    # 3. Poids selon l'âge du sondage : les plus récents comptent davantage (décroissance exponentielle)
    dates = pd.to_datetime(sondages.groupby("id_sondage")["terrain_fin"].first())
    age = (dates.max() - dates).dt.days
    poids = 0.5 ** (age / demi_vie)
    moyenne = par_sondage.mul(poids.reindex(par_sondage.index), axis=0).sum() / poids.sum()
    candidats = moyenne.rename("intentions_pct").reset_index()
    candidats = candidats.merge(table[["slug", "nom", "famille"]], on="slug", how="left")
    candidats["cle_nom"] = candidats["nom"].map(normaliser_nom)
    familles = candidats.groupby("famille")["intentions_pct"].sum()
    print(f"{par_sondage.shape[0]} sondages, {len(grille)} hypothèses gardées ({len(hyp_retirees)} retirées), "
          f"du {sondages['terrain_fin'].min()} au {sondages['terrain_fin'].max()}, demi-vie {demi_vie} jours")
    return candidats, familles / familles.sum()


# Fonction qui projette le vote par département pour une élection future (ou passée, pour le test)
# Entrée : grille des parts au 1er tour, parts nationales, parts nationales visées par famille,
#          année visée, amortissement (0 : écart du dernier scrutin gardé tel quel ;
#          1 : pente prolongée entièrement), années d'historique utilisées
# Sortie : une ligne par département et famille (écart du dernier scrutin, pente par an, écart projeté, part projetée)
def projeter_departements(parts_dep, parts_nat, intentions_famille, annee_cible=2027, amortissement=1.0,
                          annees=ANNEES):
    df = parts_dep[parts_dep["annee"].isin(annees)].merge(parts_nat, on=["annee", "famille"], how="left")
    df = df.fillna({"part_nationale": 0})
    df["ecart"] = df["part_exprimes"] - df["part_nationale"]
    annees = np.array(sorted(annees))
    derniere = annees.max()
    # régression linéaire de l'écart sur les années, pour toutes les paires (département, famille) d'un coup
    ecarts_large = df.pivot_table(index=["code_departement", "famille"], columns="annee", values="ecart", fill_value=0)
    pentes = np.polyfit(annees, ecarts_large[annees].T.values, 1)[0]
    ecarts = pd.DataFrame({"ecart_dernier": ecarts_large[derniere], "pente_par_an": pentes},
                          index=ecarts_large.index).reset_index()
    ecarts["ecart_projete"] = ecarts["ecart_dernier"] + amortissement * ecarts["pente_par_an"] * (annee_cible - derniere)
    ecarts["intention_nationale"] = ecarts["famille"].map(intentions_famille).fillna(0)
    ecarts["part_brute"] = (ecarts["intention_nationale"] + ecarts["ecart_projete"]).clip(lower=0)
    ecarts.loc[ecarts["intention_nationale"] == 0, "part_brute"] = 0   # famille sans candidat
    total = ecarts.groupby("code_departement")["part_brute"].transform("sum")
    ecarts["part_projetee"] = ecarts["part_brute"] / total            # chaque département remis à 100 %
    return ecarts.drop(columns="part_brute")


# Fonction qui teste la méthode sur le passé : on « prévoit » une élection connue avec les élections d'avant
# Entrée : grille des parts, parts nationales, années testées, amortissements à comparer
# Sortie : DataFrame (amortissement, année testée, erreur moyenne en points, erreur maximale), métropole seulement
def tester_sur_le_passe(parts_dep, parts_nat, annees_test=(2017, 2022), amortissements=(0, 0.25, 0.5, 0.75, 1)):
    lignes = []
    for annee in annees_test:
        historique = [a for a in ANNEES if a < annee]
        # on donne au modèle le vrai résultat national : on teste seulement la répartition entre départements
        national = parts_nat[parts_nat["annee"] == annee].set_index("famille")["part_nationale"]
        reel = parts_dep[parts_dep["annee"] == annee].set_index(["code_departement", "famille"])["part_exprimes"]
        for amortissement in amortissements:
            proj = projeter_departements(parts_dep, parts_nat, national, annee, amortissement, historique)
            proj = proj[~proj["code_departement"].str.startswith("Z")].set_index(["code_departement", "famille"])
            erreur = (proj["part_projetee"] - reel.reindex(proj.index).fillna(0)).abs() * 100
            lignes.append({"amortissement": amortissement, "annee_testee": annee,
                           "erreur_moyenne_points": erreur.mean(), "erreur_max_points": erreur.max()})
    return pd.DataFrame(lignes)


# Fonction qui choisit l'amortissement qui se trompe le moins sur le passé
# Entrée : résultats de tester_sur_le_passe
# Sortie : l'amortissement retenu, tableau des erreurs moyennes par amortissement
def choisir_amortissement(test):
    resume = test.pivot_table(index="amortissement", columns="annee_testee", values="erreur_moyenne_points")
    resume["moyenne"] = resume.mean(axis=1)
    return resume["moyenne"].idxmin(), resume


# Fonction qui calcule la tendance 2027 par département
# Entrée : grille des parts au 1er tour, parts nationales, intentions 2027 par famille, amortissement choisi
# Sortie : une ligne par département et famille, avec part_2027
def tendance_2027(parts_dep, parts_nat, intentions_famille, amortissement=0.5):
    proj = projeter_departements(parts_dep, parts_nat, intentions_famille, 2027, amortissement)
    return proj.rename(columns={"part_projetee": "part_2027", "ecart_dernier": "ecart_2022", "ecart_projete": "ecart_2027"})


CANDIDATS_PROJETES = {"MARINE LE PEN": ("LE PEN", "extreme_droite"),
                      "EDOUARD PHILIPPE": ("PHILIPPE", "centre"),
                      "JEAN-LUC MELENCHON": ("MELENCHON", "gauche_radicale")}


# Fonction qui compare notre tendance à la projection publiée par le site Repère 2027
# Entrée : tendance par département et famille, intentions moyennes par candidat
# Sortie : comparaison par département, résumé par candidat (corrélation, écart moyen en points)
def comparer_projection_site(tendance, sondage):
    site = lire_csv_sondage("projection-departementale-1er-tour.csv")
    infos = site["candidat"].map(normaliser_nom).map(CANDIDATS_PROJETES)
    site["cle_nom"], site["famille"] = infos.str[0], infos.str[1]
    # poids du candidat dans sa famille (sondages nationaux), pour passer de la famille au candidat
    sondage = sondage.assign(poids=sondage["intentions_pct"] / sondage.groupby("famille")["intentions_pct"].transform("sum"))
    site = site.merge(sondage[["cle_nom", "poids"]], on="cle_nom", how="left")
    site = site.merge(tendance[["code_departement", "famille", "part_2027"]], on=["code_departement", "famille"], how="left")
    site["notre_estimation_pct"] = 100 * site["part_2027"] * site["poids"]
    resume = site.groupby("candidat").apply(lambda g: pd.Series({
        "correlation": g["score_estime_pct"].corr(g["notre_estimation_pct"]),
        "ecart_moyen_points": (g["notre_estimation_pct"] - g["score_estime_pct"]).abs().mean()}),
        include_groups=False)
    return site, resume

### Requête 3 bis (temporelle) : test complet sur le passé et second tour 2027

Le test précédent donne au modèle le vrai résultat national : il ne vérifie que la carte. Ici on fait un
**test complet** : on se met à la place de quelqu'un qui aurait fait notre analyse en 2017 ou en 2022,
avec seulement les sondages publiés à ce moment-là, puis on compare au vrai résultat.
- **5 mois avant le 1er tour** (120 à 200 jours) : la situation d'aujourd'hui pour 2027.
- **1 mois avant** (0 à 30 jours).

**Second tour 2027** : même méthode. Le niveau national vient de la moyenne pondérée des sondages de chaque duel
(demi-vie de 30 jours) ; la carte garde l'écart de chaque département au national du second tour 2022
(Macron contre Le Pen). On teste cette carte en prévoyant 2022 avec les écarts de 2017, et on mesure de combien
les sondages se trompaient sur Le Pen au second tour en 2017 et 2022.

Source des sondages 2017, 2022 et 2027 (1er et 2nd tours) : tableaux des pages Wikipédia « Liste de sondages sur
l'élection présidentielle française », extraits par le dépôt GitHub `hugoC1/sondages_presidentielles`
(fichiers `sondages_1er_tour.csv` et `sondages_2nd_tour.csv`, à placer dans `data/raw/`). Ce n'est pas une
source officielle : les chiffres viennent des instituts, recopiés sur Wikipédia.

In [ ]:
# Famille politique des personnes testées dans les sondages 2017 et 2022 (y compris celles qui ne se sont pas présentées)
FAMILLES_SONDAGES = {
    "ARTHAUD": "extreme_gauche", "POUTOU": "extreme_gauche", "BESANCENOT": "extreme_gauche",
    "MELENCHON": "gauche_radicale", "ROUSSEL": "gauche_radicale",
    "HIDALGO": "gauche", "HAMON": "gauche", "HOLLANDE": "gauche", "MONTEBOURG": "gauche", "FAURE": "gauche",
    "CAZENEUVE": "gauche", "TAUBIRA": "gauche", "ROYAL": "gauche", "AUBRY": "gauche", "VALLS": "gauche",
    "PEILLON": "gauche", "PINEL": "gauche",
    "JADOT": "ecologistes", "PIOLLE": "ecologistes", "ROUSSEAU": "ecologistes", "DUFLOT": "ecologistes",
    "JOLY": "ecologistes", "HULOT": "ecologistes",
    "MACRON": "centre", "BAYROU": "centre", "LAGARDE": "centre",
    "PECRESSE": "droite", "BARNIER": "droite", "BAROIN": "droite", "BERTRAND": "droite", "CANDIDAT LR": "droite",
    "CIOTTI": "droite", "DATI": "droite", "JUVIN": "droite", "RETAILLEAU": "droite", "WAUQUIEZ": "droite",
    "FILLON": "droite", "JUPPE": "droite", "SARKOZY": "droite", "COPE": "droite", "LE MAIRE": "droite",
    "POISSON": "droite",
    "DUPONT-AIGNAN": "souverainistes", "ASSELINEAU": "souverainistes", "PHILIPPOT": "souverainistes",
    "DE VILLIERS": "souverainistes",
    "LE PEN": "extreme_droite", "ZEMMOUR": "extreme_droite",
    "LASSALLE": "divers", "CHEMINADE": "divers", "THOUY": "divers", "PAYRE": "divers",
}


# Fonction qui charge les sondages 2017, 2022 et 2027 extraits de Wikipédia
# Entrée : tour (1 ou 2), dossier des données brutes
# Sortie : DataFrame, une ligne par candidat et par hypothèse (score en %, jours avant le 1er tour, date)
def charger_sondages_historiques(tour, dossier=DOSSIER_RAW):
    nom = "sondages_1er_tour.csv" if tour == 1 else "sondages_2nd_tour.csv"
    sondages = pd.read_csv(dossier / nom, parse_dates=["date"])
    sondages["famille"] = sondages["candidat"].map(lambda c: FAMILLES_SONDAGES.get(normaliser_nom(c)))
    return sondages


# Fonction qui fait la moyenne pondérée des sondages d'une élection, par famille, dans une fenêtre de temps
# Entrée : sondages du 1er tour, année de l'élection, fenêtre (jours avant le 1er tour), demi-vie en jours
# Sortie : Series famille -> part nationale (chaque hypothèse remise à 100 %), nombre de sondages utilisés
def intentions_historiques(sondages, annee, jours_min, jours_max, demi_vie=30):
    s = sondages[(sondages["election"] == annee) & (~sondages["resultat"])
                 & sondages["jours_avant_t1"].between(jours_min, jours_max)]
    hypotheses = s.groupby(["sondage_id", "hypothese_id", "famille"])["score"].sum().unstack(fill_value=0)
    hypotheses = hypotheses.div(hypotheses.sum(axis=1), axis=0)
    par_sondage = hypotheses.groupby("sondage_id").mean()          # chaque hypothèse compte pareil dans son sondage
    dates = s.groupby("sondage_id")["date"].max()
    poids = 0.5 ** ((dates.max() - dates).dt.days / demi_vie)       # les sondages récents comptent plus
    intentions = par_sondage.mul(poids, axis=0).sum() / poids.sum()
    return intentions, s["sondage_id"].nunique()


# Fonction qui fait le test complet : sondages de l'époque + carte, comparés au vrai résultat
# Entrée : grille des parts, parts nationales, sondages du 1er tour, fenêtres de temps à tester
# Sortie : une ligne par élection, fenêtre et famille principale (sondage, réel, erreur nationale, erreur par département)
def test_complet(parts_dep, parts_nat, sondages, fenetres={"5 mois avant": (120, 200), "1 mois avant": (0, 30)}):
    principales = ["extreme_droite", "gauche_radicale", "centre", "droite", "gauche"]
    lignes = []
    for annee in (2017, 2022):
        historique = [a for a in ANNEES if a < annee]
        national_reel = parts_nat[parts_nat["annee"] == annee].set_index("famille")["part_nationale"]
        reel = parts_dep[parts_dep["annee"] == annee].set_index(["code_departement", "famille"])["part_exprimes"]
        for fenetre, (jmin, jmax) in fenetres.items():
            intentions, n = intentions_historiques(sondages, annee, jmin, jmax)
            proj = projeter_departements(parts_dep, parts_nat, intentions, annee, 0, historique)
            proj = proj[~proj["code_departement"].str.startswith("Z")].set_index(["code_departement", "famille"])
            erreur = (proj["part_projetee"] - reel.reindex(proj.index).fillna(0)).abs() * 100
            en_tete_proj = proj["part_projetee"].unstack().idxmax(axis=1)
            en_tete_reel = reel.reindex(proj.index).unstack().idxmax(axis=1)
            bon_en_tete = (en_tete_proj == en_tete_reel).mean()
            for famille in principales:
                lignes.append({"annee": annee, "fenetre": fenetre, "nb_sondages": n, "famille": famille,
                               "sondage": intentions.get(famille, 0) * 100,
                               "reel": national_reel.get(famille, 0) * 100,
                               "erreur_departements_points": erreur.xs(famille, level="famille").mean(),
                               "famille_en_tete_correcte": bon_en_tete})
    test = pd.DataFrame(lignes)
    test["erreur_nationale_points"] = test["sondage"] - test["reel"]
    return test


# Fonction qui mesure l'erreur des sondages de second tour sur Le Pen (2017 et 2022)
# Entrée : sondages du 2nd tour, fenêtres de temps (jours avant le 1er tour)
# Sortie : une ligne par élection et fenêtre (score moyen de Le Pen dans les sondages, résultat réel, écart)
def biais_second_tour(sondages2, fenetres={"5 mois avant": (90, 250), "1 mois avant": (0, 30)}):
    lignes = []
    for annee in (2017, 2022):
        s = sondages2[(sondages2["election"] == annee) & (sondages2["candidat"] == "Le Pen")
                      & sondages2["duel"].str.contains("Macron")]
        reel = s.loc[s["resultat"], "score"].iloc[0]
        for fenetre, (jmin, jmax) in fenetres.items():
            f = s[(~s["resultat"]) & s["jours_avant_t1"].between(jmin, jmax)]
            lignes.append({"annee": annee, "fenetre": fenetre, "nb_sondages": f["sondage_id"].nunique(),
                           "le_pen_sondages": f["score"].mean(), "le_pen_reel": reel})
    biais = pd.DataFrame(lignes)
    biais["ecart_points"] = biais["le_pen_sondages"] - biais["le_pen_reel"]
    return biais


# Fonction qui fait la moyenne pondérée des sondages de chaque duel de second tour 2027
# Entrée : sondages du 2nd tour, candidat du RN (Le Pen ou Bardella), demi-vie en jours
# Sortie : une ligne par adversaire (nombre de sondages, score moyen du candidat RN, minimum, maximum, dernier sondage)
def duels_2027(sondages2, candidat_rn="Le Pen", demi_vie=30):
    s = sondages2[(sondages2["election"] == 2027) & (~sondages2["resultat"])]
    rn = s[s["candidat"] == candidat_rn]
    rn = rn.assign(adversaire=rn["duel"].str.replace(candidat_rn, "", regex=False).str.strip(" –"))
    poids = 0.5 ** ((s["date"].max() - rn["date"]).dt.days / demi_vie)
    rn = rn.assign(poids=poids)
    duels = rn.groupby("adversaire").apply(lambda d: pd.Series({
        "nb_sondages": d["sondage_id"].nunique(),
        "score_rn": np.average(d["score"], weights=d["poids"]),
        "min": d["score"].min(), "max": d["score"].max(), "dernier_sondage": d["date"].max().date()}),
        include_groups=False)
    return duels.sort_values("nb_sondages", ascending=False)


# Fonction qui teste la carte du second tour : on prévoit 2022 avec les écarts de 2017 et le vrai national
# Entrée : voix par famille et département (avec le 2nd tour)
# Sortie : erreur moyenne en points, part des départements où le bon candidat arrive en tête
def tester_carte_second_tour(familles_dep):
    t2 = familles_dep[(familles_dep["tour"] == 2) & (familles_dep["famille"] == "extreme_droite")
                      & ~familles_dep["code_departement"].str.startswith("Z")]
    part = t2.pivot(index="code_departement", columns="annee", values="part_exprimes").dropna(subset=[2017, 2022])
    national = t2.groupby("annee")["voix"].sum() / t2.groupby("annee")["exprimes"].sum()
    prevu = part[2017] - national[2017] + national[2022]
    erreur = (prevu - part[2022]).abs().mean() * 100
    bon_gagnant = ((prevu > 0.5) == (part[2022] > 0.5)).mean()
    return erreur, bon_gagnant


# Fonction qui projette le second tour 2027 par département
# Entrée : voix par famille et département, moyennes des duels (score du candidat RN en %)
# Sortie : une ligne par département et adversaire (part du candidat RN)
def second_tour_2027(familles_dep, duels):
    t2 = familles_dep[(familles_dep["annee"] == 2022) & (familles_dep["tour"] == 2)
                      & (familles_dep["famille"] == "extreme_droite")]
    national_2022 = t2["voix"].sum() / t2["exprimes"].sum()
    ecart = t2.set_index("code_departement")["part_exprimes"] - national_2022
    lignes = [pd.DataFrame({"code_departement": ecart.index, "adversaire": adversaire,
                            "part_rn": (ecart.values + score / 100).clip(0.01, 0.99)})
              for adversaire, score in duels["score_rn"].items()]
    return pd.concat(lignes, ignore_index=True)

### Requête 4 (spatiale) : clustering des départements

On regroupe les 96 départements de métropole selon leur **profil de vote au 1er tour 2022** (part de chaque
famille et abstention), avec **K-means** sur des variables standardisées. Le nombre de groupes est choisi par le
**score de silhouette** (entre 3 et 6 groupes). On affiche ensuite les groupes sur la carte : s'ils forment des
zones contiguës, la géographie du vote est structurée dans l'espace. Chaque groupe est décrit par les familles
qui y sont le plus au-dessus de leur moyenne, et par ses indicateurs socio-économiques moyens.

**Clustering spatial.** Ce premier découpage n'utilise pas la position des départements : ses groupes peuvent être
éparpillés. On ajoute donc un **clustering spatial** : K-means sur le profil de vote **et** sur la position du
département (latitude et longitude de son centre, calculées à partir des communes), la position pesant autant que
le profil de vote. On compare les deux découpages avec la **silhouette** et la **cohérence spatiale** (part des
4 départements voisins qui sont dans le même groupe).

In [ ]:
# Fonction qui choisit le nombre de groupes et applique K-means
# Entrée : table des départements, colonnes du profil, nombres de groupes à tester
# Sortie : table avec une colonne cluster, DataFrame des scores de silhouette
def clustering_departements(departements, colonnes, k_possibles=range(3, 7)):
    X = StandardScaler().fit_transform(departements[colonnes])
    scores = {k: silhouette_score(X, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X))
              for k in k_possibles}
    meilleur_k = max(scores, key=scores.get)
    groupes = KMeans(n_clusters=meilleur_k, n_init=10, random_state=0).fit_predict(X)
    resultat = departements.assign(cluster=groupes)
    return resultat, pd.Series(scores, name="silhouette").rename_axis("k").to_frame()


LIBELLES = {"vote_extreme_gauche": "extrême gauche", "vote_gauche_radicale": "gauche radicale",
            "vote_gauche": "gauche", "vote_ecologistes": "écologistes", "vote_centre": "centre",
            "vote_droite": "droite", "vote_souverainistes": "souverainistes",
            "vote_extreme_droite": "extrême droite", "taux_abstention": "abstention"}


# Fonction qui décrit chaque groupe et lui donne un nom lisible
# Entrée : table avec la colonne cluster, colonnes du profil de vote, indicateurs socio-économiques
# Sortie : table avec la colonne nom_cluster, DataFrame des profils moyens par groupe
def decrire_clusters(departements, colonnes, indicateurs):
    moyennes = departements.groupby("cluster")[colonnes + indicateurs].mean()
    ecarts = moyennes[colonnes] - departements[colonnes].mean()      # écart à la moyenne des départements
    noms = {}
    for cluster, ligne in ecarts.iterrows():
        deux_premiers = ligne.sort_values(ascending=False).index[:2]
        noms[cluster] = f"Groupe {cluster + 1} : " + " et ".join(LIBELLES[c] for c in deux_premiers)
    moyennes.insert(0, "nb_departements", departements["cluster"].value_counts())
    moyennes.index = moyennes.index.map(noms)
    return departements.assign(nom_cluster=departements["cluster"].map(noms)), moyennes


# Fonction qui dessine une carte des départements
# Entrée : DataFrame avec code_departement, colonne à colorer, titre, contours (chargés si absents)
# Sortie : figure Plotly (choroplèthe, métropole seulement)
def carte_departements(df, colonne, titre, contours=None):
    contours = contours or charger_contours()
    codes_metropole = {f["properties"]["code"] for f in contours["features"]}
    df = df[df["code_departement"].isin(codes_metropole)]
    continu = pd.api.types.is_numeric_dtype(df[colonne])
    fig = px.choropleth(df, geojson=contours, locations="code_departement", featureidkey="properties.code",
                        color=colonne, hover_name="code_departement", title=titre,
                        color_continuous_scale="Viridis" if continu else None)
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig

# Fonction qui mesure la cohérence spatiale d'un découpage en groupes
# Entrée : table avec latitude, longitude et une colonne de groupe, nombre de voisins regardés
# Sortie : part des voisins (les k départements les plus proches) qui sont dans le même groupe, entre 0 et 1
def coherence_spatiale(departements, colonne="cluster", k=4):
    coordonnees = np.radians(departements[["latitude", "longitude"]].values)
    voisins = NearestNeighbors(n_neighbors=k + 1, metric="haversine").fit(coordonnees)
    _, indices = voisins.kneighbors(coordonnees)                 # le premier voisin est le département lui-même
    groupes = departements[colonne].values
    return (groupes[indices[:, 1:]] == groupes[:, None]).mean()


# Fonction qui fait un clustering spatial : K-means sur la position du département ET son profil de vote
# Entrée : table des départements (avec latitude et longitude), colonnes du profil, poids de la position,
#          nombres de groupes à tester
# Sortie : table avec une colonne cluster, DataFrame (k, silhouette, cohérence spatiale)
def clustering_spatial(departements, colonnes, poids_position=1.0, k_possibles=range(3, 9)):
    profil = StandardScaler().fit_transform(departements[colonnes])
    position = StandardScaler().fit_transform(departements[["latitude", "longitude"]])
    # la position (2 colonnes) reçoit autant de poids que tout le profil de vote (len(colonnes) colonnes)
    position = position * np.sqrt(len(colonnes) / 2) * poids_position
    X = np.hstack([profil, position])
    lignes = []
    for k in k_possibles:
        groupes = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
        lignes.append({"k": k, "silhouette": silhouette_score(X, groupes),
                       "coherence_spatiale": coherence_spatiale(departements.assign(g=groupes), "g")})
    scores = pd.DataFrame(lignes).set_index("k")
    meilleur_k = scores["silhouette"].idxmax()
    groupes = KMeans(n_clusters=meilleur_k, n_init=10, random_state=0).fit_predict(X)
    return departements.assign(cluster=groupes), scores

## Étape 6 : interprétation et dashboard

Le dashboard Dash (`dashboard/app.py`) reprend les quatre requêtes : cartes et évolutions par famille
(requête 1), règles d'association (requête 2), tendance 2027 (requête 3) et groupes de départements
(requête 4). Il lit les fichiers enregistrés dans `data/processed/` par `main()`.

In [ ]:
# Fonction qui lance le dashboard
# Entrée : mode d'affichage ("external" : lien à ouvrir dans le navigateur ; "inline" : dans le notebook)
# Sortie : aucune (le serveur Dash tourne jusqu'à l'arrêt du noyau)
def lancer_dashboard(mode="external", port=8050):
    import sys
    sys.path.insert(0, str(DOSSIER_PROJET / "dashboard"))
    from app import creer_app
    creer_app().run(jupyter_mode=mode, port=port, debug=False, threaded=False)

## Main

In [ ]:
# Fonction principale : enchaîne toutes les étapes du KDD, enregistre les résultats et lance le dashboard
# Entrée : lancer (True pour lancer le dashboard à la fin)
# Sortie : dictionnaire des principaux résultats
def main(lancer=True):
    DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

    # Étape 1 : sélection des données
    generaux = ajouter_annee_tour(charger_resultats_generaux())
    candidats = ajouter_annee_tour(charger_resultats_candidats())
    sondages = charger_sondages()
    sondages_t1, sondages_t2 = charger_sondages_historiques(1), charger_sondages_historiques(2)
    communes, filosofi, acled = charger_communes(), charger_filosofi(), charger_acled()

    # Étape 2 : exploration
    for df, nom in [(generaux, "Résultats généraux"), (candidats, "Résultats par candidat"),
                    (sondages, "Sondages 2027"), (communes, "Communes 2026"), (filosofi, "Filosofi 2023"),
                    (acled, "ACLED"), (sondages_t1, "Sondages 1er tour 2017-2027"),
                    (sondages_t2, "Sondages 2nd tour 2017-2027")]:
        display(explorer(df, nom))

    # Étape 3 : prétraitement
    candidats = ajouter_familles(candidats, charger_familles())

    # Étape 4 : transformation
    generaux_dep = agreger_par_departement(generaux)
    familles_dep = voix_par_famille_departement(candidats, generaux_dep)
    parts_dep = parts_premier_tour(familles_dep)
    parts_nat = parts_nationales(candidats, generaux)
    acled_region = acled_par_region(acled, "2021-04-10", "2022-04-10")   # l'année avant le 1er tour 2022
    departements = table_departements(familles_dep, generaux_dep, profil_departements(communes),
                                      filosofi, acled_region)
    print("Clés communes : départements qui trouvent une correspondance dans chaque source :")
    display(verifier_cles(generaux_dep, profil_departements(communes), filosofi, acled, charger_contours()))
    generaux_dep.to_parquet(DOSSIER_PROCESSED / "participation_departements.parquet", index=False)
    familles_dep.to_parquet(DOSSIER_PROCESSED / "familles_departements.parquet", index=False)

    # Requête 1 : regroupements
    display(resume_national(candidats))
    sondage, intentions_famille = moyenne_sondages(sondages, candidats_retires())
    evolution = evolution_nationale(parts_nat, intentions_famille)
    evolution.to_parquet(DOSSIER_PROCESSED / "evolution_nationale.parquet")
    graphique_evolution(evolution).show()
    print("Abstention nationale :")
    display(evolution_abstention(generaux_dep).round(3))
    print("Extrême droite, 2017 -> 2022 : plus fortes et plus faibles évolutions (points) :")
    display(departements_qui_bougent(parts_dep, "extreme_droite")["evolution_points"].round(1))
    indicateurs = ["niveau_vie_median", "taux_pauvrete", "indice_gini", "part_revenu_chomage",
                   "densite", "part_population_rurale", "evenements_protests_pour_100k"]
    for cible in ["vote_extreme_droite", "vote_gauche_radicale", "vote_centre", "taux_abstention"]:
        r2, coefficients = modele_explicatif(departements, cible, indicateurs)
        print(f"{cible} : R² = {r2:.2f} ; effet d'un écart-type de chaque indicateur (points) :")
        display(coefficients.round(2))
    ipc = charger_ipc()
    print("Inflation sur 12 mois en mars de chaque année d'élection :")
    display(ipc.loc[[f"{a}-03-01" for a in ANNEES], "inflation_12_mois"].round(3))

    # Requête 2 : motifs fréquents et règles d'association
    regles = regles_association(construire_transactions(departements, parts_dep))
    regles.to_parquet(DOSSIER_PROCESSED / "regles_association.parquet", index=False)
    print(f"{len(regles)} règles d'association retenues :")
    display(regles.head(10).round(2))

    # Requête 3 : tendance 2027, après un test de la méthode sur 2017 et 2022
    test = tester_sur_le_passe(parts_dep, parts_nat)
    amortissement, erreurs = choisir_amortissement(test)
    print(f"Test sur le passé : erreur moyenne (points) selon l'amortissement ; retenu : {amortissement}")
    display(erreurs.round(2))
    test.to_parquet(DOSSIER_PROCESSED / "test_sur_le_passe.parquet", index=False)
    tendance = tendance_2027(parts_dep, parts_nat, intentions_famille, amortissement)
    tendance.to_parquet(DOSSIER_PROCESSED / "tendance_2027_departements.parquet", index=False)
    print("Intentions 2027 par famille (moyenne des sondages) :")
    display(intentions_famille.sort_values(ascending=False).round(3))
    _, comparaison = comparer_projection_site(tendance, sondage)
    print("Comparaison avec la projection du site (par département) :")
    display(comparaison.round(2))
    carte_departements(tendance[tendance["famille"] == "extreme_droite"], "part_2027",
                       "Tendance 2027 : extrême droite au 1er tour").show()

    # Table unique : toutes les sources réunies, une ligne par département et par tour
    table_unique = construire_table_unique(familles_dep, generaux_dep, profil_departements(communes), filosofi,
                                           acled, ipc, tendance)
    table_unique.to_parquet(DOSSIER_PROCESSED / "table_unique.parquet", index=False)
    print(f"Table unique : {table_unique.shape[0]} lignes (département x tour), {table_unique.shape[1]} colonnes")
    display(explorer(table_unique, "Table unique"))
    display(table_unique[table_unique["id_election"] == "2022_pres_t1"].head())

    # Requête 3 bis : test complet sur le passé (sondages de l'époque + carte) et second tour 2027
    complet = test_complet(parts_dep, parts_nat, sondages_t1)
    complet.to_parquet(DOSSIER_PROCESSED / "test_complet.parquet", index=False)
    print("Test complet, 1er tour : sondages de l'époque contre résultat réel (%), erreur par département (points) :")
    display(complet.round(1))
    biais = biais_second_tour(sondages_t2)
    print("Second tour : score de Le Pen dans les sondages et dans les urnes (%) :")
    display(biais.round(1))
    erreur_carte, bon_gagnant = tester_carte_second_tour(familles_dep)
    print(f"Carte du second tour testée sur 2022 : erreur moyenne {erreur_carte:.1f} points, "
          f"bon gagnant dans {bon_gagnant:.0%} des départements")
    duels = duels_2027(sondages_t2)
    print("Second tour 2027 : moyenne pondérée des sondages de chaque duel (score de Le Pen, %) :")
    display(duels.round(1))
    duels.reset_index().to_parquet(DOSSIER_PROCESSED / "duels_2027.parquet", index=False)
    second_tour = second_tour_2027(familles_dep, duels[duels["nb_sondages"] >= 10])
    second_tour.to_parquet(DOSSIER_PROCESSED / "second_tour_2027.parquet", index=False)
    philippe = second_tour[second_tour["adversaire"] == "Philippe"]
    philippe = philippe[~philippe["code_departement"].str.startswith("Z")]
    print(f"Le Pen contre Philippe : Le Pen en tête dans {(philippe['part_rn'] > 0.5).sum()} départements sur {len(philippe)}")
    carte_departements(philippe, "part_rn", "Second tour 2027 : score de Le Pen contre Philippe").show()

    # Requête 4 : clustering des départements, sur le vote seul puis sur le vote et la position (clustering spatial)
    profil_vote = [c for c in departements.columns if c.startswith("vote_") and c != "vote_divers"] + ["taux_abstention"]
    departements, silhouettes = clustering_departements(departements, profil_vote)
    print("Score de silhouette selon le nombre de groupes :")
    display(silhouettes.round(3))
    departements, profils = decrire_clusters(departements, profil_vote, indicateurs)
    display(profils.round(3))
    carte_departements(departements, "nom_cluster", "Groupes de départements (profil de vote 2022)").show()
    spatial, scores_spatial = clustering_spatial(departements.drop(columns=["cluster", "nom_cluster"]), profil_vote)
    print("Clustering spatial (vote + position) : silhouette et cohérence spatiale selon le nombre de groupes :")
    display(scores_spatial.round(3))
    spatial, profils_spatial = decrire_clusters(spatial, profil_vote, indicateurs)
    display(profils_spatial.round(3))
    print(f"Cohérence spatiale : {coherence_spatiale(departements):.2f} avec le vote seul, "
          f"{coherence_spatiale(spatial):.2f} avec le vote et la position")
    departements["cluster_spatial"] = spatial["cluster"].values
    departements["nom_cluster_spatial"] = spatial["nom_cluster"].values
    carte_departements(departements, "nom_cluster_spatial", "Clustering spatial (vote 2022 + position)").show()
    departements.to_parquet(DOSSIER_PROCESSED / "departements_2022.parquet", index=False)

    # Étape 6 : dashboard
    if lancer:
        lancer_dashboard()
    return {"departements": departements, "tendance": tendance, "regles": regles, "evolution": evolution,
            "test_complet": complet, "second_tour": second_tour,
            "table_unique": table_unique}


if __name__ == "__main__":
    # LANCER_DASHBOARD=0 permet d'exécuter le notebook sans lancer le serveur (tests automatiques)
    resultats = main(lancer=os.environ.get("LANCER_DASHBOARD", "1") == "1")